In [1]:
from utils.orthanc_utils import *
from utils.db_utils import *

def get_volumes(dicoms):    
    df = pd.DataFrame(
        [{"SliceLocation": d.SliceLocation, "InstanceNumber": d.InstanceNumber, "PixelArray": d.pixel_array} for d in dicoms]
    ).sort_values(["SliceLocation", "InstanceNumber"])

    d0 = dicoms[0]
    pixel_spacing = float(d0.PixelSpacing[0])
    thickness = float(getattr(d0, "SpacingBetweenSlices", getattr(d0, "SliceThickness", np.nan)))
    voxel_size = (pixel_spacing ** 2 * thickness) / 1000
    
    array_4d = []
    for _, slice_df in df.groupby("SliceLocation"):
        time_array = []
        for _, time_df in slice_df.groupby("InstanceNumber"):
            pixel_array = time_df["PixelArray"].iloc[0]
            time_array.append(pixel_array)

        slice_array = np.stack(time_array, axis=-1)
        array_4d.append(slice_array)

    array_4d = np.stack(array_4d, axis=-2)

    labels = {500: "lv", 1000: "rv", 1500: "lv_myo", 2000: "rv_myo"}
    curves = {
        name: np.sum(array_4d == val, axis=(0, 1, 2)) * voxel_size
        for val, name in labels.items()
    }
    lv, rv = curves["lv"], curves["rv"]
    lv_edv, lv_esv = lv.max(), lv.min()
    rv_edv, rv_esv = rv.max(), rv.min()

    metrics = {
        "lv_edv": lv_edv,
        "lv_esv": lv_esv,
        "lv_ef": (lv_edv - lv_esv) / lv_edv,
        "lv_mass": np.mean(curves["lv_myo"]) * 1.05,
        "rv_edv": rv_edv,
        "rv_esv": rv_esv,
        "rv_ef": (rv_edv - rv_esv) / rv_edv,
        "rv_mass": np.mean(curves["rv_myo"]) * 1.05,
    }

    return metrics

METRICS_PATH = 'clasp_metrics.csv'

db = TinyDB('image_clasp_db.json')
StudyQuery = Query()
SeriesQuery = Query()

metrics_df = pd.read_csv(METRICS_PATH)

rows = []
for study in db.search(StudyQuery.series.any(SeriesQuery.dl_orthanc_id != None)):
    dl_processed_series_list = [series for series in study.get("series", []) if series.get("dl_orthanc_id") != None]

    image_dicoms = [d for s in dl_processed_series_list for d in fetch_dicoms_for_series(s['orthanc_series_id'])]
    mask_dicoms = [d for s in dl_processed_series_list for d in fetch_dicoms_for_series(s['dl_orthanc_id'])]
    metrics = get_volumes(mask_dicoms)
    metrics['orthanc_study_id'] = study['orthanc_study_id']
    rows.append(metrics)

    masked_images = [image.pixel_array * (mask.pixel_array > 0) for image, mask in zip(image_dicoms, mask_dicoms) ]
    new_orthanc_id = send_series_to_orthanc(masked_images, image_dicoms, new_description='Roundel Processed Series')

metrics_df = pd.concat([metrics_df, pd.DataFrame(rows)], ignore_index=True).set_index('orthanc_study_id')
metrics_df.to_csv(METRICS_PATH)


/tmp/ipykernel_163452/2090013513.py:69: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  metrics_df = pd.concat([metrics_df, pd.DataFrame(rows)], ignore_index=True).set_index('orthanc_study_id')
